In [1]:
import pandas as pd
import numpy as np
import requests
import streamlit as st
import sqlite3

In [68]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

def scrape_cbb_data(season=2026, table_type="school_advanced"):
    """
    Valid table_types:
    - 'school_basic'
    - 'school_advanced'
    - 'opponent_basic'
    - 'opponent_advanced'
    """
    
    # 1. Map the table type to the exact URL configuration and HTML element ID
    config = {
        "school_basic": {
            "url_suffix": "school-stats.html",
            "table_id": "basic_school_stats"
        },
        "school_advanced": {
            "url_suffix": "advanced-school-stats.html",
            "table_id": "adv_school_stats"
        },
        "opponent_basic": {
            "url_suffix": "opponent-stats.html",
            "table_id": "basic_opp_stats"
        },
        "opponent_advanced": {
            "url_suffix": "advanced-opponent-stats.html",
            "table_id": "adv_opp_stats"
        }
    }
    
    if table_type not in config:
        raise ValueError(f"Invalid table_type. Choose from: {list(config.keys())}")
        
    # 2. Build the URL dynamically
    suffix = config[table_type]["url_suffix"]
    target_table_id = config[table_type]["table_id"]
    url = f"https://www.sports-reference.com/cbb/seasons/men/{season}-{suffix}"
    
    # 3. Fetch the web page
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed to fetch data. Status code: {response.status_code}")
        return None
        
    # 4. Parse HTML and find the specific table ID
    soup = BeautifulSoup(response.content, 'html.parser')
    table = soup.find('table', {'id': target_table_id})
    
    if table is None:
        print(f"Error: Could not find table with ID '{target_table_id}' on the page.")
        return None
        
    # 5. Read into Pandas
    df = pd.read_html(StringIO(str(table)))[0]
    
    # 6. Clean Multi-index headers
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join(col).strip() for col in df.columns.values]
        
    # 7. Remove repeated header rows throughout the table
    school_col = [c for c in df.columns if 'School' in c][0]
    df = df[df[school_col] != 'School']
    df = df.reset_index(drop=True)
    
    return df

# --- HOW TO RUN IT FOR ALL 4 TABLES ---

tables_to_get = ["school_basic", "school_advanced", "opponent_basic", "opponent_advanced"]

for t_type in tables_to_get:
    print(f"Fetching {t_type} data...")
    
    # Scrape the data
    data = scrape_cbb_data(season=2026, table_type=t_type)
    
    if data is not None:
        # Generate a distinct filename for each table
        filename = f"cbb_{t_type}_2026.csv"
        data.to_csv(filename, index=False)
        print(f"--> Saved {data.shape[0]} teams to {filename}")
        
    # CRITICAL: Sleep 4 seconds between tables to respect the 20 requests per minute rule!
    print("Sleeping 4 seconds to respect rate limits...")
    time.sleep(4)

print("\nAll downloads complete! Check your folder for the 4 CSV files.")

Fetching school_basic data...
--> Saved 383 teams to cbb_school_basic_2026.csv
Sleeping 4 seconds to respect rate limits...
Fetching school_advanced data...
--> Saved 383 teams to cbb_school_advanced_2026.csv
Sleeping 4 seconds to respect rate limits...


KeyboardInterrupt: 

In [2]:
df_sa = pd.read_csv("data/2026/cbb_school_advanced_2026.csv")
df_sb = pd.read_csv("data/2026/cbb_school_basic_2026.csv")
df_oa = pd.read_csv("data/2026/cbb_opponent_advanced_2026.csv")
df_ob = pd.read_csv("data/2026/cbb_opponent_basic_2026.csv")
df_ob.head()

,Unnamed: 0_level_0_Rk,Unnamed: 1_level_0_School,Overall_G,Overall_W,Overall_L,Overall_W-L%,Overall_SRS,Overall_SOS,Unnamed: 8_level_0_Unnamed: 8_level_1,Conf._W,...,Opponent_FT,Opponent_FTA,Opponent_FT%,Opponent_ORB,Opponent_TRB,Opponent_AST,Opponent_STL,Opponent_BLK,Opponent_TOV,Opponent_PF
0,1.0,Abilene Christian,33,14,19,.424,-7.38,-1.45,NaN,5,...,607,842,.721,266,1029,393,278,147,541,605
1,2.0,Air Force,32,3,29,.094,-13.75,4.60,NaN,0,...,488,671,.727,321,1146,488,255,113,325,569
2,3.0,Akron NCAA,35,29,6,.829,8.48,-2.91,NaN,17,...,444,616,.721,383,1157,467,221,100,470,547
3,4.0,Alabama NCAA,35,25,10,.714,23.03,14.57,NaN,13,...,535,769,.696,483,1395,488,224,133,334,675
4,5.0,Alabama A&M,33,18,15,.545,-13.31,-10.81,NaN,10,...,518,714,.725,329,1129,463,202,99,372,634


In [3]:
data = ["df_sa", "df_sb"]
cols = ["Opponent_FGA", "Opponent_TOV", "Opponent_ORB", "Opponent_FTA", "Points_Opp."]
totals = ["Totals_FGA", "Totals_TOV", "Totals_ORB", "Totals_FTA", "Points_Tm."]


for i in cols:
    df_ob[i] = pd.to_numeric(df_ob[i], errors='coerce')
for i in totals:
    df_sb[i] = pd.to_numeric(df_sb[i], errors='coerce')

In [4]:
df_ob["Possessions"] = df_ob["Opponent_FGA"] + df_ob["Opponent_TOV"] - df_ob["Opponent_ORB"] + (.475 * df_ob["Opponent_FTA"])
#df_ob.head()
df_ob["Def_Eff"] = df_ob["Points_Opp."] / df_ob["Possessions"]
df_ob.head()

,Unnamed: 0_level_0_Rk,Unnamed: 1_level_0_School,Overall_G,Overall_W,Overall_L,Overall_W-L%,Overall_SRS,Overall_SOS,Unnamed: 8_level_0_Unnamed: 8_level_1,Conf._W,...,Opponent_FT%,Opponent_ORB,Opponent_TRB,Opponent_AST,Opponent_STL,Opponent_BLK,Opponent_TOV,Opponent_PF,Possessions,Def_Eff
0,1.0,Abilene Christian,33,14,19,.424,-7.38,-1.45,NaN,5,...,.721,266.0,1029,393,278,147,541.0,605,2295.950,1.040528
1,2.0,Air Force,32,3,29,.094,-13.75,4.60,NaN,0,...,.727,321.0,1146,488,255,113,325.0,569,2156.725,1.185130
2,3.0,Akron NCAA,35,29,6,.829,8.48,-2.91,NaN,17,...,.721,383.0,1157,467,221,100,470.0,547,2506.600,1.033272
3,4.0,Alabama NCAA,35,25,10,.714,23.03,14.57,NaN,13,...,.696,483.0,1395,488,224,133,334.0,675,2617.275,1.106494
4,5.0,Alabama A&M,33,18,15,.545,-13.31,-10.81,NaN,10,...,.725,329.0,1129,463,202,99,372.0,634,2242.150,1.067279


In [5]:
df_sb["Possessions"] = df_sb["Totals_FGA"] + df_sb["Totals_TOV"] - df_sb["Totals_ORB"] + (.475 * df_sb["Totals_FTA"])
df_sb["Off_Eff"] = df_sb["Points_Tm."] / df_sb["Possessions"]
df_sb["Overall_G"] = pd.to_numeric(df_sb["Overall_G"], errors='coerce')
df_sb["Poss_Per_Game"] = df_sb["Possessions"] / df_sb["Overall_G"]
df_sb.head()

,Unnamed: 0_level_0_Rk,Unnamed: 1_level_0_School,Overall_G,Overall_W,Overall_L,Overall_W-L%,Overall_SRS,Overall_SOS,Unnamed: 8_level_0_Unnamed: 8_level_1,Conf._W,...,Totals_ORB,Totals_TRB,Totals_AST,Totals_STL,Totals_BLK,Totals_TOV,Totals_PF,Possessions,Off_Eff,Poss_Per_Game
0,1.0,Abilene Christian,33.0,14,19,.424,-7.38,-1.45,NaN,5,...,369.0,1047,440,321,91,472.0,691,2294.975,1.011340,69.544697
1,2.0,Air Force,32.0,3,29,.094,-13.75,4.60,NaN,0,...,225.0,931,389,180,76,472.0,570,2156.650,0.912990,67.395313
2,3.0,Akron NCAA,35.0,29,6,.829,8.48,-2.91,NaN,17,...,397.0,1319,639,261,109,381.0,549,2505.975,1.227067,71.599286
3,4.0,Alabama NCAA,35.0,25,10,.714,23.03,14.57,NaN,13,...,437.0,1425,571,225,174,343.0,646,2613.200,1.221491,74.662857
4,5.0,Alabama A&M,33.0,18,15,.545,-13.31,-10.81,NaN,10,...,311.0,1126,387,198,93,376.0,607,2246.800,1.058394,68.084848


In [6]:
df_combine = pd.DataFrame({
    'Team': df_sb['Unnamed: 1_level_0_School'],
    'Offensive': round(df_sb['Off_Eff'] * 100, 2),
    'Defensive': round(df_ob['Def_Eff'] * 100, 2),
    'Possessions': df_sb['Poss_Per_Game']
})

df_combine

,Team,Offensive,Defensive,Possessions
0,Abilene Christian,101.13,104.05,69.544697
1,Air Force,91.30,118.51,67.395313
2,Akron NCAA,122.71,103.33,71.599286
3,Alabama NCAA,122.15,110.65,74.662857
4,Alabama A&M,105.84,106.73,68.084848
...,...,...,...,...
378,Wright State NCAA,116.50,106.47,69.111429
379,Wyoming,113.20,107.94,67.914394
380,Xavier,109.42,112.18,71.616667
381,Yale,121.75,105.91,66.770161


In [7]:
df_combine['Team'] = df_combine['Team'].str.replace(r'\s*NCAA$', '', regex=True)

# Also clear out any asterisks if Sports-Reference added them
df_combine['Team'] = df_combine['Team'].str.replace(r'\*$', '', regex=True)

# Finally, strip any accidental trailing/leading spaces left over
df_combine['Team'] = df_combine['Team'].str.strip()

# Check your clean school names
print(df_combine['Team'].head(10))

0    Abilene Christian
1            Air Force
2                Akron
3              Alabama
4          Alabama A&M
5        Alabama State
6          Albany (NY)
7         Alcorn State
8             American
9    Appalachian State
Name: Team, dtype: object


In [8]:
df_combine

,Team,Offensive,Defensive,Possessions
0,Abilene Christian,101.13,104.05,69.544697
1,Air Force,91.30,118.51,67.395313
2,Akron,122.71,103.33,71.599286
3,Alabama,122.15,110.65,74.662857
4,Alabama A&M,105.84,106.73,68.084848
...,...,...,...,...
378,Wright State,116.50,106.47,69.111429
379,Wyoming,113.20,107.94,67.914394
380,Xavier,109.42,112.18,71.616667
381,Yale,121.75,105.91,66.770161


In [9]:
df_sb["Overall_SOS"] = pd.to_numeric(df_sb["Overall_SOS"], errors='coerce')

In [12]:
df_combine["Offensive"] = round(df_sb["Off_Eff"] * ((df_sb["Overall_SOS"]/200) + 1) * 100,2)
df_combine["Defensive"] = round(df_ob["Def_Eff"] * ((df_sb["Overall_SOS"]/200) - 1) * -100,2)
df_combine["Net"] = round(df_combine["Offensive"] - df_combine["Defensive"],2)
df_combine["Possessions"] = round(df_combine["Possessions"], 2)
df_combine.insert(0, "Net", df_combine.pop("Net"))
df_combine.insert(0, "Team", df_combine.pop("Team"))
df_combine.head()

,Team,Net,Offensive,Defensive,Possessions
0,Abilene Christian,-4.41,100.40,104.81,69.54
1,Air Force,-22.39,93.40,115.79,67.40
2,Akron,16.09,120.92,104.83,71.60
3,Alabama,28.46,131.05,102.59,74.66
4,Alabama A&M,-12.38,100.12,112.50,68.08


In [14]:
df_combine.to_csv('output2.csv', index=False)

In [34]:
import cbbpy.mens_scraper as m
m.get_game_pbp('401522202')

""


In [33]:
import cbbpy.womens_scraper as w
w.get_team_schedule('davidson', 2025)

""


In [30]:
import cbbpy.womens_scraper as w

# Capitalized exact match
df = w.get_team_schedule('Davidson', 2022)

# Alternative if the above fails
df_wildcats = w.get_team_schedule('Davidson Wildcats', 2022)


No exact match for 'Davidson Wildcats'. Fetching closest team match: 'Davidson'.


In [32]:
df_wildcats

""


In [42]:
import requests

# The active Game ID found by your previous script run
game_id = "401856600"

# This endpoint is significantly less strict and specifically serves play-by-play
url = f"https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/events/{game_id}/competitions/{game_id}/plays?limit=500"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

print(f"Attempting to pull PBP from mobile core API for Game ID: {game_id}...")
response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    # This endpoint returns a dictionary where the list of plays lives in 'items'
    plays = data.get('items', [])
    
    if plays:
        print(f"\n--- Success! Pulled {len(plays)} plays ---")
        for play in plays[0:]:  # Print the first 5 plays to verify data shape
            clock = play.get('clock', {}).get('displayValue', '0:00')
            text = play.get('text', '[No Description]')
            
            # The score is usually nested inside a period/homeScore array here
            print(f"[{clock}] {text}")
    else:
        print("\nConnected, but the 'items' list containing the plays was empty.")
else:
    print(f"\nFailed again. HTTP Status Code: {response.status_code}")
    print("ESPN core API rejected this specific URL formatting.")

Attempting to pull PBP from mobile core API for Game ID: 401856600...

--- Success! Pulled 482 plays ---
[20:00] Start game
[19:56] Jump Ball won by UConn
[19:56] Jump Ball lost by Michigan
[19:34] Tarris Reed Jr. misses 10-foot hook shot
[19:30] Elliot Cadeau Defensive Rebound.
[19:18] Morez Johnson Jr. misses layup
[19:16] Michigan Offensive Rebound.
[19:01] Aday Mara misses 9-foot turnaround jump shot
[18:57] Tarris Reed Jr. Defensive Rebound.
[18:45] Braylon Mullins misses 26-foot three point jumper
[18:41] Morez Johnson Jr. Defensive Rebound.
[18:23] Morez Johnson Jr. makes layup (Elliot Cadeau assists)
[17:57] Tarris Reed Jr. makes 7-foot hook shot
[17:36] Elliot Cadeau makes layup
[17:16] Tarris Reed Jr. misses 8-foot turnaround jump shot
[17:12] Elliot Cadeau Defensive Rebound.
[16:55] Foul on Silas Demary Jr..
[16:55] Elliot Cadeau makes free throw 1 of 3
[16:55] Elliot Cadeau makes free throw 2 of 3
[16:55] Aday Mara subbing out for Michigan
[16:55] Roddy Gayle Jr. subbing in

In [ ]:
def process_game_into_stints(pbp_items, home_starters, away_starters):
    """
    Parses raw ESPN play-by-play items to map exactly who is on the floor
    and records 'stints' whenever a substitution or period end occurs.
    """
    # Initialize the active lineups with the starters
    current_home = set(home_starters)
    current_away = set(away_starters)
    
    stints_log = []
    
    # Track the scoreboard at the start of the current stint
    stint_start_home_score = 0
    stint_start_away_score = 0
    stint_start_clock = "20:00"
    
    for play in pbp_items:
        text = play.get('text', '')
        clock = play.get('clock', {}).get('displayValue', '0:00')
        home_score = play.get('homeScore', stint_start_home_score)
        away_score = play.get('awayScore', stint_start_away_score)
        
        # Detect a substitution
        # ESPN text style: "John Doe subbed in for Smith, Jane." or "John Doe enters the game for Jane Smith"
        if "subbing in for" in text:
            
            # 1. Calculate what happened during the stint that just ended
            home_delta = home_score - stint_start_home_score
            away_delta = away_score - stint_start_away_score
            
            stints_log.append({
                "start_clock": stint_start_clock,
                "end_clock": clock,
                "home_lineup": list(current_home),
                "away_lineup": list(current_away),
                "home_point_margin": home_delta - away_delta,
                # You will add possession-estimation logic here later
            })
            
            # 2. Update the active lineups based on the text
            # (In production, you'll use Regex to extract player names from the string)
            # Example placeholder logic:
            # current_home.remove(player_out)
            # current_home.add(player_in)
            
            # 3. Reset the baseline for the next stint
            stint_start_home_score = home_score
            stint_start_away_score = away_score
            stint_start_clock = clock
            
    return stints_log

In [46]:
import requests

def get_game_starters_v3(game_id):
    """
    Queries ESPN's web-native CDN endpoint to safely extract starters
    without triggering dynamic parameter filtering.
    """
    # Using the clean XHR web endpoint directly
    url = f"https://site.web.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary?event={game_id}"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    
    response = requests.get(url, headers=headers)
    
    home_starters = []
    away_starters = []
    
    if response.status_code == 200:
        data = response.json()
        boxscore = data.get('boxscore', {})
        players_list = boxscore.get('players', [])
        
        # Guard against empty payloads
        if not players_list:
            print("Connected, but player stat blocks were empty.")
            return [], []
            
        for team_index, team_data in enumerate(players_list):
            # ESPN usually puts Away Team at Index 0 and Home Team at Index 1
            statistics = team_data.get('statistics', [])
            if not statistics:
                continue
                
            # The athletes list lives inside the first stats object array
            athletes = statistics[0].get('athletes', [])
            
            for athlete in athletes:
                # If 'starter' flag is True, grab their name
                if athlete.get('starter') == True:
                    name = athlete.get('athlete', {}).get('displayName', 'Unknown')
                    
                    if team_index == 1:
                        home_starters.append(name)
                    else:
                        away_starters.append(name)
                        
        return home_starters, away_starters
    else:
        print(f"CDN Request failed. HTTP Status Code: {response.status_code}")
        return [], []

# --- Run the Execution ---
game_id = "401856600"
home, away = get_game_starters_v3(game_id)

print(f"Game ID: {game_id}")
print(f"Away Starters: {away}")
print(f"Home Starters: {home}")



Game ID: 401856600
Away Starters: ['Alex Karaban', 'Tarris Reed Jr.', 'Braylon Mullins', 'Silas Demary Jr.', 'Solo Ball']
Home Starters: ['Yaxel Lendeborg', 'Morez Johnson Jr.', 'Aday Mara', 'Elliot Cadeau', 'Nimari Burnett']


In [48]:
process_game_into_stints(pbp_items, ['Yaxel Lendeborg', 'Morez Johnson Jr.', 'Aday Mara', 'Elliot Cadeau', 'Nimari Burnett'], ['Alex Karaban', 'Tarris Reed Jr.', 'Braylon Mullins', 'Silas Demary Jr.', 'Solo Ball'])

NameError: name 'pbp_items' is not defined

In [57]:
import requests
import re

# --- 1. THE LINEUP TRACKER FUNCTION ---
import copy
import re

def process_game_with_stats(pbp_items, home_starters, away_starters, home_team="UConn", away_team="Michigan"):
    current_home = set(home_starters)
    current_away = set(away_starters)
    
    stints_log = []
    stint_start_clock = "20:00"
    
    stats = {
        "home": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0},
        "away": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0}
    }
    
    for play in pbp_items:
        text = play.get('text', '')
        clock = play.get('clock', {}).get('displayValue', '0:00')
        text_lower = text.lower()
        
        is_home = home_team.lower() in text_lower
        is_away = away_team.lower() in text_lower
        act_team = "home" if is_home else "away" if is_away else None
        
        # --- Accumulate live stats ---
        if act_team:
            if "makes" in text_lower or "misses" in text_lower:
                if "free throw" not in text_lower:
                    stats[act_team]["fga"] += 1
                if "makes" in text_lower:
                    if "three-point" in text_lower or "3-point" in text_lower:
                        stats[act_team]["pts"] += 3
                    elif "free throw" in text_lower:
                        stats[act_team]["pts"] += 1
                        stats[act_team]["fta"] += 1
                    else:
                        stats[act_team]["pts"] += 2
                elif "misses" in text_lower and "free throw" in text_lower:
                    stats[act_team]["fta"] += 1
            elif "turnover" in text_lower or "bad pass" in text_lower or "lost ball" in text_lower:
                stats[act_team]["tov"] += 1
            elif "offensive rebound" in text_lower:
                stats[act_team]["orb"] += 1

        # --- Substitution Event ---
        if "subbing in" in text_lower or "subbing out" in text_lower or "enters the game" in text_lower:
            home_poss = calculate_possessions(stats["home"])
            away_poss = calculate_possessions(stats["away"])
            
            # CRITICAL FIX: Use deepcopy so the inner dictionary values are frozen in time
            stints_log.append({
                "start_clock": stint_start_clock,
                "end_clock": clock,
                "home_lineup": list(current_home),
                "away_lineup": list(current_away),
                "stats": copy.deepcopy(stats), 
                "home_possessions": home_poss,
                "away_possessions": away_poss
            })
            
            # Update lineups
            match_in = re.search(r"(.+?)\s+(?:subbing in|enters the game)", text)
            match_out = re.search(r"(.+?)\s+(?:subbing out|leaves the game)", text)
            
            if match_in and act_team:
                if act_team == "home": current_home.add(match_in.group(1).strip())
                else: current_away.add(match_in.group(1).strip())
            if match_out:
                p_out = match_out.group(1).strip()
                current_home.discard(p_out)
                current_away.discard(p_out)
            
            # Reset counters for the next lineup window
            stats = {
                "home": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0},
                "away": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0}
            }
            stint_start_clock = clock
            
    return stints_log

from collections import defaultdict

def calculate_and_print_player_efficiencies(calculated_stints):
    player_stats = defaultdict(lambda: {"off_pts": 0, "off_poss": 0, "def_pts": 0, "def_poss": 0})
    
    for stint in calculated_stints:
        # Extra defensive check: skip if stats didn't log properly for some reason
        if "stats" not in stint or not stint["stats"]:
            continue
            
        home_lineup = stint["home_lineup"]
        away_lineup = stint["away_lineup"]
        
        home_pts = stint["stats"]["home"]["pts"]
        away_pts = stint["stats"]["away"]["pts"]
        
        home_poss = stint["home_possessions"]
        away_poss = stint["away_possessions"]
        
        # Credit Home Players
        for player in home_lineup:
            player_stats[player]["off_pts"] += home_pts
            player_stats[player]["off_poss"] += home_poss
            player_stats[player]["def_pts"] += away_pts
            player_stats[player]["def_poss"] += away_poss
            
        # Credit Away Players
        for player in away_lineup:
            player_stats[player]["off_pts"] += away_pts
            player_stats[player]["off_poss"] += away_poss
            player_stats[player]["def_pts"] += home_pts
            player_stats[player]["def_poss"] += home_poss

    # Print block
    print(f"\n{'Player Name':<25} | {'OFF PTS':<7} | {'OFF POSS':<8} | {'ORtg':<6} || {'DRtg':<6}")
    print("-"*75)
    for player, s in sorted(player_stats.items(), key=lambda x: x[1]['off_poss'], reverse=True):
        if not player or player == "Unknown" or len(player) < 3:
            continue
        ortg = (s["off_pts"] / s["off_poss"] * 100) if s["off_poss"] > 0 else 0.0
        drtg = (s["def_pts"] / s["def_poss"] * 100) if s["def_poss"] > 0 else 0.0
        print(f"{player:<25} | {s['off_pts']:<7} | {s['off_poss']:<8.2f} | {ortg:<6.1f} || {drtg:<6.1f}")

# --- 2. GET THE STARTERS ---
def get_game_starters(game_id):
    url = f"https://site.web.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary?event={game_id}"
    headers = {"User-Agent": "Mozilla/5.0"}
    res = requests.get(url, headers=headers)
    home, away = [], []
    if res.status_code == 200:
        players_list = res.json().get('boxscore', {}).get('players', [])
        for index, team_data in enumerate(players_list):
            athletes = team_data.get('statistics', [{}])[0].get('athletes', [])
            for athlete in athletes:
                if athlete.get('starter') == True:
                    name = athlete.get('athlete', {}).get('displayName', 'Unknown')
                    if index == 1: home.append(name)
                    else: away.append(name)
    return home, away

# --- 3. PIPELINE EXECUTION ---
game_id = "401856600"
headers = {"User-Agent": "Mozilla/5.0"}

# Step A: Get the Starters
home_lineup, away_lineup = get_game_starters(game_id)

# Step B: Get the Play-by-Play items payload
pbp_url = f"https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/events/{game_id}/competitions/{game_id}/plays?limit=500"
pbp_response = requests.get(pbp_url, headers=headers)

if pbp_response.status_code == 200:
    # THIS KEY IS WHAT YOU ARE ASSIGNING TO pbp_items
    raw_plays_list = pbp_response.json().get('items', [])
    
    # Step C: Run the combined tracking matrix
    calculated_stints = process_game_into_stints(raw_plays_list, home_lineup, away_lineup)
    
    print(f"Successfully split game into {len(calculated_stints)} structural line-up stints!")
    # Show the first tracked stint as a test preview
    if calculated_stints:
        print("\n--- Example Stint Captured ---")
        print(f"Stint Window: {calculated_stints[0]['start_clock']} down to {calculated_stints[0]['end_clock']}")
        print(f"Score Margin During Stint: {calculated_stints[0]['net_point_margin']}")
        print(f"Sub Trigger Text: {calculated_stints[0]['raw_text_trigger']}")
else:
    print("Could not retrieve play log items.")

Successfully split game into 66 structural line-up stints!

--- Example Stint Captured ---
Stint Window: 20:00 down to 16:55
Score Margin During Stint: 4
Sub Trigger Text: Roddy Gayle Jr. subbing in for Michigan


In [69]:
import requests
import re
import copy
from collections import defaultdict
import pandas as pd

def calculate_possessions(s):
    return s["fga"] + (0.475 * s["fta"]) + s["tov"] - s["orb"]

def get_game_data_and_player_map(game_id):
    """
    Fetches starters, team names, and builds a direct dictionary mapping 
    every player name to their respective squad (home or away).
    """
    url = f"https://site.web.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary?event={game_id}"
    headers = {"User-Agent": "Mozilla/5.0"}
    res = requests.get(url, headers=headers)
    
    home_starters, away_starters = [], []
    player_team_map = {} # Maps "Player Name" -> "home" or "away"
    
    if res.status_code == 200:
        data = res.json()
        players_list = data.get('boxscore', {}).get('players', [])
        
        for index, team_data in enumerate(players_list):
            # Index 1 is Home, Index 0 is Away
            team_side = "home" if index == 1 else "away"
            athletes = team_data.get('statistics', [{}])[0].get('athletes', [])
            
            for athlete in athletes:
                name = athlete.get('athlete', {}).get('displayName', 'Unknown')
                if name == 'Unknown':
                    continue
                    
                # Map this player permanently to their home/away side
                player_team_map[name] = team_side
                
                # Double check starters
                if athlete.get('starter') == True:
                    if team_side == "home":
                        home_starters.append(name)
                    else:
                        away_starters.append(name)
                        
    return home_starters, away_starters, player_team_map

def process_game_with_stats(pbp_items, home_starters, away_starters, player_team_map):
    current_home = set(home_starters)
    current_away = set(away_starters)
    stints_log = []
    stint_start_clock = "20:00"
    
    stats = {
        "home": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0},
        "away": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0}
    }
    
    # Combined master roster list to easily identify which player is mentioned in the text row
    all_players = list(player_team_map.keys())
    
    for play in pbp_items:
        text = play.get('text', '')
        clock = play.get('clock', {}).get('displayValue', '0:00')
        text_lower = text.lower()
        
        # --- NEW & IMPROVED TEAM IDENTIFICATION ---
        # Find which player is mentioned anywhere in this play text string
        act_team = None
        for player in all_players:
            if player in text:
                act_team = player_team_map[player]
                break # Found the player executing the play, stop looking
        
        # If no explicit player name matched, check for a general team keyword fallback
        if not act_team:
            if "michigan" in text_lower: act_team = "away"
            elif "uconn" in text_lower or "connecticut" in text_lower: act_team = "home"

        # --- STATISTIC ACCUMULATION ---
        if act_team:
            is_shot = any(kw in text_lower for kw in ["shot", "layup", "jumper", "dunk", "three-pointer", "3-pointer", "free throw"])
            is_make = any(kw in text_lower for kw in ["makes", "made", "good"])
            is_miss = any(kw in text_lower for kw in ["misses", "missed"])
            
            if is_shot:
                if "free throw" not in text_lower:
                    stats[act_team]["fga"] += 1
                if is_make:
                    if "free throw" in text_lower:
                        stats[act_team]["pts"] += 1
                        stats[act_team]["fta"] += 1
                    elif "three-point" in text_lower or "3-point" in text_lower or "three-pointer" in text_lower:
                        stats[act_team]["pts"] += 3
                    else:
                        stats[act_team]["pts"] += 2
                elif is_miss and "free throw" in text_lower:
                    stats[act_team]["fta"] += 1
            elif any(kw in text_lower for kw in ["turnover", "bad pass", "lost ball", "traveling", "offensive foul"]):
                stats[act_team]["tov"] += 1
            elif "offensive rebound" in text_lower or "off rebound" in text_lower:
                stats[act_team]["orb"] += 1

        # --- LINEUP SUB TRIGGER EVENT ---
        if "subbing in" in text_lower or "subbing out" in text_lower or "enters the game" in text_lower:
            home_poss = calculate_possessions(stats["home"])
            away_poss = calculate_possessions(stats["away"])
            
            stints_log.append({
                "start_clock": stint_start_clock,
                "end_clock": clock,
                "home_lineup": list(current_home),
                "away_lineup": list(current_away),
                "stats": copy.deepcopy(stats), 
                "home_possessions": home_poss,
                "away_possessions": away_poss
            })
            
            match_in = re.search(r"(.+?)\s+(?:subbing in|enters the game)", text)
            match_out = re.search(r"(.+?)\s+(?:subbing out|leaves the game)", text)
            
            if match_in and act_team:
                p_in = match_in.group(1).strip()
                if act_team == "home": current_home.add(p_in)
                else: current_away.add(p_in)
            if match_out:
                p_out = match_out.group(1).strip()
                current_home.discard(p_out)
                current_away.discard(p_out)
            
            stats = {
                "home": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0},
                "away": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0}
            }
            stint_start_clock = clock
            
    return stints_log

import pandas as pd

def calculate_and_get_player_dfs(calculated_stints, player_team_map):
    player_stats = defaultdict(lambda: {"off_pts": 0, "off_poss": 0, "def_pts": 0, "def_poss": 0})
    
    # 1. Aggregate stats from stints
    for stint in calculated_stints:
        home_lineup = stint["home_lineup"]
        away_lineup = stint["away_lineup"]
        
        home_pts = stint["stats"]["home"]["pts"]
        away_pts = stint["stats"]["away"]["pts"]
        home_poss = stint["home_possessions"]
        away_poss = stint["away_possessions"]
        
        for player in home_lineup:
            player_stats[player]["off_pts"] += home_pts
            player_stats[player]["off_poss"] += home_poss
            player_stats[player]["def_pts"] += away_pts
            player_stats[player]["def_poss"] += away_poss
            
        for player in away_lineup:
            player_stats[player]["off_pts"] += away_pts
            player_stats[player]["off_poss"] += away_poss
            player_stats[player]["def_pts"] += home_pts
            player_stats[player]["def_poss"] += home_poss

    # 2. Build a raw list of rows for the DataFrame
    rows = []
    for player, s in player_stats.items():
        if not player or player == "Unknown" or len(player) < 3:
            continue
            
        ortg = (s["off_pts"] / s["off_poss"] * 100) if s["off_poss"] > 0 else 0.0
        drtg = (s["def_pts"] / s["def_poss"] * 100) if s["def_poss"] > 0 else 0.0
        
        rows.append({
            "Player": player,
            "Team_Side": player_team_map.get(player, "home"),
            "Off_Pts": s["off_pts"],
            "Off_Poss": round(s["off_poss"], 2),
            "ORtg": round(ortg, 1),
            "Def_Pts": s["def_pts"],
            "Def_Poss": round(s["def_poss"], 2),
            "DRtg": round(drtg, 1),
            "Net_Rating": round(ortg - drtg, 1) # Added Net Rating for analytics depth
        })
        
    # 3. Create Master DataFrame and split by Team Side
    df_master = pd.DataFrame(rows)
    
    home_df = df_master[df_master["Team_Side"] == "home"].sort_values(by="Off_Poss", ascending=False).reset_index(drop=True)
    away_df = df_master[df_master["Team_Side"] == "away"].sort_values(by="Off_Poss", ascending=False).reset_index(drop=True)
    
    # Drop the structural column for clean printing
    home_df = home_df.drop(columns=["Team_Side"])
    away_df = away_df.drop(columns=["Team_Side"])
    
    # 4. Display them inside the console beautifully
    print("\n" + "="*80)
    print(f"{'HOME TEAM DATAFRAME':^80}")
    print("="*80)
    print(home_df.to_string(index=False))
    
    print("\n" + "="*80)
    print(f"{'AWAY TEAM DATAFRAME':^80}")
    print("="*80)
    print(away_df.to_string(index=False))
    print("="*80)
    
    return home_df, away_df

# --- PIPELINE ENGINE ---
game_id = "401825468"
headers = {"User-Agent": "Mozilla/5.0"}

# 1. Fetch data structures cleanly
home_l, away_l, player_map = get_game_data_and_player_map(game_id)
print(f"Loaded database map containing {len(player_map)} active players.")

# 2. Extract play logs
pbp_url = f"https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/events/{game_id}/competitions/{game_id}/plays?limit=500"
pbp_response = requests.get(pbp_url, headers=headers)

if pbp_response.status_code == 200:
    raw_plays_list = pbp_response.json().get('items', [])
    
    # 3. Calculate stint-by-stint matrix
    calculated_stints = process_game_with_stats(raw_plays_list, home_l, away_l, player_map)
    
    # 4. Stream final analytical chart
    home_efficiency_df, away_efficiency_df = calculate_and_get_player_dfs(calculated_stints, player_map)
    #home_efficiency_df.to_csv("home_stints.csv", index=False)
else:
    print("Could not retrieve play log items.")

Loaded database map containing 30 active players.

                              HOME TEAM DATAFRAME                               
           Player  Off_Pts  Off_Poss  ORtg  Def_Pts  Def_Poss  DRtg  Net_Rating
     Braden Smith       70     56.75 123.3       55     55.02 100.0        23.4
      Oscar Cluff       45     43.22 104.1       47     46.65 100.8         3.4
Trey Kaufman-Renn       43     39.38 109.2       34     38.80  87.6        21.6
         C.J. Cox       40     38.80 103.1       41     41.65  98.4         4.7
       Omer Mayer       34     31.38 108.4       38     32.55 116.7        -8.4
   Fletcher Loyer       36     30.32 118.7       28     29.95  93.5        25.2
      Jack Benter       30     22.85 131.3       27     22.23 121.5         9.8
   Gicarri Harris       31     21.43 144.7       25     21.80 114.7        30.0
  Daniel Jacobsen       26     17.00 152.9       15     13.85 108.3        44.6

                              AWAY TEAM DATAFRAME                  

In [ ]:
import requests
import re
import copy
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.linear_model import Ridge

# =====================================================================
# 1. MATHEMATICAL & POSSESSION HELPERS
# =====================================================================
def calculate_possessions(s):
    return s["fga"] + (0.475 * s["fta"]) + s["tov"] - s["orb"]

def check_kenpom_garbage_time(clock_string, period, home_score, away_score):
    """
    Applies KenPom's phased threshold based on the actual cumulative game score
    and the TRUE total minutes remaining in the contest.
    """
    try:
        minutes, seconds = map(int, clock_string.split(":"))
        half_mins_remaining = minutes + (seconds / 60.0)
        
        # --- FIXED: Adjust time remaining based on the current half ---
        if period == 1:
            mins_remaining = half_mins_remaining + 20.0 # 1st half counts down to 20 mins remaining
        else:
            mins_remaining = half_mins_remaining # 2nd half counts down to 0 mins remaining
    except:
        return False
        
    margin = abs(home_score - away_score)
    
    if mins_remaining > 10.0:
        if margin >= 30: return True
    elif 5.0 <= mins_remaining <= 10.0:
        if margin >= 25: return True
    elif 2.0 <= mins_remaining < 5.0:
        if margin >= 20: return True
    elif mins_remaining < 2.0:
        if margin >= 11: return True
        
    return False


# =====================================================================
# 2. DATA ACQUISITION & TEAM MAPPING
# =====================================================================
def get_game_data_and_player_map(game_id):
    url = f"https://site.web.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary?event={game_id}"
    headers = {"User-Agent": "Mozilla/5.0"}
    res = requests.get(url, headers=headers)
    
    home_starters, away_starters = [], []
    player_team_map = {}
    
    if res.status_code == 200:
        data = res.json()
        players_list = data.get('boxscore', {}).get('players', [])
        
        for index, team_data in enumerate(players_list):
            team_side = "home" if index == 1 else "away"
            athletes = team_data.get('statistics', [{}])[0].get('athletes', [])
            
            for athlete in athletes:
                name = athlete.get('athlete', {}).get('displayName', 'Unknown')
                if name == 'Unknown':
                    continue
                player_team_map[name] = team_side
                if athlete.get('starter') == True:
                    if team_side == "home": home_starters.append(name)
                    else: away_starters.append(name)
                        
    return home_starters, away_starters, player_team_map


# =====================================================================
# 3. ROLLING PLAY-BY-PLAY STINT PARSER
# =====================================================================
def process_game_with_stats(pbp_items, home_starters, away_starters, player_team_map):
    current_home = set(home_starters)
    current_away = set(away_starters)
    stints_log = []
    stint_start_clock = "20:00"
    
    running_home_score = 0
    running_away_score = 0
    
    stats = {
        "home": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0},
        "away": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0}
    }
    
    all_players = list(player_team_map.keys())
    
    for play in pbp_items:
        text = play.get('text', '')
        clock = play.get('clock', {}).get('displayValue', '0:00')
        text_lower = text.lower()
        
        # --- NEW: Safely extract the current game period from the ESPN play dictionary ---
        period = play.get('period', {}).get('number', 1)
        
        act_team = None
        for player in all_players:
            if player in text:
                act_team = player_team_map[player]
                break
        
        if not act_team:
            if "michigan" in text_lower or "rutgers" in text_lower: act_team = "away"
            elif "uconn" in text_lower or "illinois" in text_lower: act_team = "home"

        if act_team:
            is_shot = any(kw in text_lower for kw in ["shot", "layup", "jumper", "dunk", "three-pointer", "3-pointer", "free throw"])
            is_make = any(kw in text_lower for kw in ["makes", "made", "good"])
            is_miss = any(kw in text_lower for kw in ["misses", "missed"])
            
            if is_shot:
                if "free throw" not in text_lower:
                    stats[act_team]["fga"] += 1
                if is_make:
                    if "free throw" in text_lower:
                        stats[act_team]["pts"] += 1
                        running_home_score += (1 if act_team == "home" else 0)
                        running_away_score += (1 if act_team == "away" else 0)
                        stats[act_team]["fta"] += 1
                    elif "three-point" in text_lower or "3-point" in text_lower or "three-pointer" in text_lower:
                        stats[act_team]["pts"] += 3
                        running_home_score += (3 if act_team == "home" else 0)
                        running_away_score += (3 if act_team == "away" else 0)
                    else:
                        stats[act_team]["pts"] += 2
                        running_home_score += (2 if act_team == "home" else 0)
                        running_away_score += (2 if act_team == "away" else 0)
                elif is_miss and "free throw" in text_lower:
                    stats[act_team]["fta"] += 1
            elif any(kw in text_lower for kw in ["turnover", "bad pass", "lost ball", "traveling", "offensive foul"]):
                stats[act_team]["tov"] += 1
            elif "offensive rebound" in text_lower or "off rebound" in text_lower:
                stats[act_team]["orb"] += 1

        # Substitution Stint Trigger Check
        if "subbing in" in text_lower or "subbing out" in text_lower or "enters the game" in text_lower:
            
            # --- FIXED: Now passes the accurate period parameter ---
            if check_kenpom_garbage_time(clock, period, running_home_score, running_away_score):
                print(f"\n>>> [@ H{period} {clock}] Match hit true KenPom garbage threshold ({abs(running_home_score - running_away_score)}pt margin). Halting data tracking.")
                break
                
            home_poss = calculate_possessions(stats["home"])
            away_poss = calculate_possessions(stats["away"])
            
            stints_log.append({
                "start_clock": stint_start_clock,
                "end_clock": clock,
                "home_lineup": list(current_home),
                "away_lineup": list(current_away),
                "stats": copy.deepcopy(stats), 
                "home_possessions": home_poss,
                "away_possessions": away_poss
            })
            
            match_in = re.search(r"(.+?)\s+(?:subbing in|enters the game)", text)
            match_out = re.search(r"(.+?)\s+(?:subbing out|leaves the game)", text)
            
            if match_in and act_team:
                p_in = match_in.group(1).strip()
                if act_team == "home": current_home.add(p_in)
                else: current_away.add(p_in)
            if match_out:
                p_out = match_out.group(1).strip()
                current_home.discard(p_out)
                current_away.discard(p_out)
            
            stats = {
                "home": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0},
                "away": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0}
            }
            stint_start_clock = clock
            
    return stints_log


# =====================================================================
# 4. BAYESIAN REGULARIZED LINEUP MODEL (RAPM) MATRICES
# =====================================================================
def build_rapm_matrices(calculated_stints, player_team_map):
    all_players = sorted(list(player_team_map.keys()))
    player_to_idx = {player: idx for idx, player in enumerate(all_players)}
    
    matrix_rows = []
    targets = []
    weights = []
    
    for stint in calculated_stints:
        home_poss = stint["home_possessions"]
        away_poss = stint["away_possessions"]
        total_poss = (home_poss + away_poss) / 2
        
        if total_poss <= 0:
            continue
            
        home_pts = stint["stats"]["home"]["pts"]
        away_pts = stint["stats"]["away"]["pts"]
        
        home_ortg = (home_pts / home_poss * 100) if home_poss > 0 else 0
        away_ortg = (away_pts / away_poss * 100) if away_poss > 0 else 0
        stint_net_rating = home_ortg - away_ortg
        
        row = np.zeros(len(all_players))
        
        for player in stint["home_lineup"]:
            if player in player_to_idx: row[player_to_idx[player]] = 1
        for player in stint["away_lineup"]:
            if player in player_to_idx: row[player_to_idx[player]] = -1
                
        matrix_rows.append(row)
        targets.append(stint_net_rating)
        weights.append(total_poss)
        
    return np.array(matrix_rows), np.array(targets), np.array(weights), all_players

def train_rapm_model(X, Y, W, player_names, alpha=10):
    model = Ridge(alpha=alpha, fit_intercept=False)
    model.fit(X, Y, sample_weight=W)
    return pd.DataFrame({"Player": player_names, "Bayesian_RAPM": np.round(model.coef_, 2)})


# =====================================================================
# 5. INTEGRATED DATAFRAME GENERATOR, MERGE, & EXPORT
# =====================================================================
def generate_master_analytics_and_export(calculated_stints, player_team_map, rapm_df, game_id):
    player_stats = defaultdict(lambda: {"off_pts": 0, "off_poss": 0, "def_pts": 0, "def_poss": 0})
    
    for stint in calculated_stints:
        home_lineup = stint["home_lineup"]
        away_lineup = stint["away_lineup"]
        
        home_pts = stint["stats"]["home"]["pts"]
        away_pts = stint["stats"]["away"]["pts"]
        home_poss = stint["home_possessions"]
        away_poss = stint["away_possessions"]
        
        for player in home_lineup:
            player_stats[player]["off_pts"] += home_pts
            player_stats[player]["off_poss"] += home_poss
            player_stats[player]["def_pts"] += away_pts
            player_stats[player]["def_poss"] += away_poss
            
        for player in away_lineup:
            player_stats[player]["off_pts"] += away_pts
            player_stats[player]["off_poss"] += away_poss
            player_stats[player]["def_pts"] += home_pts
            player_stats[player]["def_poss"] += home_poss

    rows = []
    for player, s in player_stats.items():
        if not player or player == "Unknown" or len(player) < 3:
            continue
            
        ortg = (s["off_pts"] / s["off_poss"] * 100) if s["off_poss"] > 0 else 0.0
        drtg = (s["def_pts"] / s["def_poss"] * 100) if s["def_poss"] > 0 else 0.0
        
        rows.append({
            "Player": player,
            "Team_Side": player_team_map.get(player, "home"),
            "Off_Pts": s["off_pts"],
            "Off_Poss": round(s["off_poss"], 2),
            "ORtg": round(ortg, 1),
            "Def_Pts": s["def_pts"],
            "Def_Poss": round(s["def_poss"], 2),
            "DRtg": round(drtg, 1),
            "Net_Rating": round(ortg - drtg, 1)
        })
        
    df_boxscore = pd.DataFrame(rows)
    df_master = pd.merge(df_boxscore, rapm_df, on="Player", how="left")
    
    home_df = df_master[df_master["Team_Side"] == "home"].sort_values(by="Bayesian_RAPM", ascending=False).reset_index(drop=True)
    away_df = df_master[df_master["Team_Side"] == "away"].sort_values(by="Bayesian_RAPM", ascending=False).reset_index(drop=True)
    
    print("\n" + "="*110)
    print(f"{'HOME TEAM METRIC SHEET (FILTERED COMPETITIVE POSSESSIONS)':^110}")
    print("="*110)
    print(home_df.drop(columns=["Team_Side"]).to_string(index=False))
    
    print("\n" + "="*110)
    print(f"{'AWAY TEAM METRIC SHEET (FILTERED COMPETITIVE POSSESSIONS)':^110}")
    print("="*110)
    print(away_df.drop(columns=["Team_Side"]).to_string(index=False))
    print("="*110)
    
    output_filename = f"game_{game_id}_analytics_matrix.csv"
    #df_master.to_csv(output_filename, index=False)
    print(f"\n[SUCCESS] Clean competitive dataset exported to '{output_filename}'")
    
    return home_df, away_df


# =====================================================================
# 6. CENTRAL EXECUTOR
# =====================================================================
if __name__ == "__main__":
    game_id = "401856566" 
    headers = {"User-Agent": "Mozilla/5.0"}
    
    print("Initializing engine...")
    home_l, away_l, player_map = get_game_data_and_player_map(game_id)
    print(f"Roster mapped successfully. Loaded {len(player_map)} unique players.")
    
    pbp_url = f"https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/events/{game_id}/competitions/{game_id}/plays?limit=500"
    pbp_response = requests.get(pbp_url, headers=headers)
    
    if pbp_response.status_code == 200:
        raw_plays_list = pbp_response.json().get('items', [])
        
        calculated_stints = process_game_with_stats(raw_plays_list, home_l, away_l, player_map)
        X, Y, W, player_names = build_rapm_matrices(calculated_stints, player_map)
        rapm_leaderboard = train_rapm_model(X, Y, W, player_names, alpha=10)
        
        home_final_df, away_final_df = generate_master_analytics_and_export(
            calculated_stints, player_map, rapm_leaderboard, game_id
        )
    else:
        print(f"Failed to access network assets. Status: {pbp_response.status_code}")

Initializing engine...
Roster mapped successfully. Loaded 28 unique players.

                          HOME TEAM METRIC SHEET (FILTERED COMPETITIVE POSSESSIONS)                           
           Player  Off_Pts  Off_Poss  ORtg  Def_Pts  Def_Poss  DRtg  Net_Rating  Bayesian_RAPM
       Milos Uzan       38     48.95  77.6       43     51.23  83.9        -6.3          15.74
    Emanuel Sharp       42     52.95  79.3       47     53.23  88.3        -9.0          12.07
    Joseph Tugler       24     33.00  72.7       29     34.90  83.1       -10.4           2.79
     Kalifa Sakho        8     12.00  66.7       11     12.43  88.5       -21.9           0.92
  Chris Cenac Jr.       28     36.95  75.8       32     37.33  85.7       -10.0          -2.39
Kingston Flemings       40     49.95  80.1       49     49.23  99.5       -19.5         -14.37
    Chase McCarty       24     24.95  96.2       30     27.80 107.9       -11.7         -16.43
     Mercy Miller        6     11.00  54.5       14

In [20]:
import requests
import re
import copy
import pandas as pd
import concurrent.futures
import time
from datetime import datetime, timedelta
from collections import defaultdict

# =====================================================================
# 1. MATHEMATICAL & POSSESSION HELPERS
# =====================================================================
def calculate_possessions(s):
    return s["fga"] + (0.475 * s["fta"]) + s["tov"] - s["orb"]

def check_kenpom_garbage_time(clock_string, period, home_score, away_score):
    try:
        minutes, seconds = map(int, clock_string.split(":"))
        half_mins_remaining = minutes + (seconds / 60.0)
        
        if period == 1:
            mins_remaining = half_mins_remaining + 20.0 
        else:
            mins_remaining = half_mins_remaining 
    except:
        return False
        
    margin = abs(home_score - away_score)
    
    if mins_remaining > 10.0:
        if margin >= 30: return True
    elif 5.0 <= mins_remaining <= 10.0:
        if margin >= 25: return True
    elif 2.0 <= mins_remaining < 5.0:
        if margin >= 20: return True
    elif mins_remaining < 2.0:
        if margin >= 11: return True
        
    return False


# =====================================================================
# 2. AUTOMATED DISCOVERY & DATA ACQUISITION (STRICT D1 FILTER)
# =====================================================================
def get_game_ids_for_date(date_str):
    """
    Fetches the day's slate and filters out any games involving non-D1 teams.
    """
    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard?dates={date_str}&groups=50&limit=1000"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    valid_game_ids = []
    
    for attempt in range(3):
        try:
            res = requests.get(url, headers=headers, timeout=10)
            if res.status_code == 200:
                events = res.json().get('events', [])
                
                for event in events:
                    competitions = event.get('competitions', [])
                    if not competitions:
                        continue
                    competitors = competitions[0].get('competitors', [])
                    if len(competitors) < 2:
                        continue
                    
                    # Extract conference metadata for both sides
                    team_a_conf = competitors[0].get('team', {}).get('conferenceId')
                    team_b_conf = competitors[1].get('team', {}).get('conferenceId')
                    
                    # FIX: If either team lacks a D1 conference ID or belongs to '51' (Non-D1 bucket), skip entirely
                    if not team_a_conf or not team_b_conf:
                        continue
                    if team_a_conf == '51' or team_b_conf == '51':
                        continue
                        
                    valid_game_ids.append(event.get('id'))
                return valid_game_ids
            elif res.status_code == 429:
                time.sleep(2)
        except:
            time.sleep(1)
            
    return valid_game_ids

def get_game_data_and_player_map(game_id):
    url = f"https://site.web.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary?event={game_id}"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    
    home_starters, away_starters = [], []
    player_team_map = {}
    home_team_name, away_team_name = "home", "away"
    
    try:
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code == 200:
            data = res.json()
            
            teams_meta = data.get('boxscore', {}).get('teams', [])
            if len(teams_meta) >= 2:
                away_team_name = teams_meta[0].get('team', {}).get('displayName', 'away').lower()
                home_team_name = teams_meta[1].get('team', {}).get('displayName', 'home').lower()
            
            players_list = data.get('boxscore', {}).get('players', [])
            for index, team_data in enumerate(players_list):
                team_side = "home" if index == 1 else "away"
                athletes = team_data.get('statistics', [{}])[0].get('athletes', [])
                
                for athlete in athletes:
                    name = athlete.get('athlete', {}).get('displayName', 'Unknown')
                    if name == 'Unknown':
                        continue
                    player_team_map[name] = team_side
                    if athlete.get('starter') == True:
                        if team_side == "home": home_starters.append(name)
                        else: away_starters.append(name)
    except:
        pass
                        
    return home_starters, away_starters, player_team_map, home_team_name, away_team_name


# =====================================================================
# 3. UNIVERSAL PLAY-BY-PLAY STINT PARSER
# =====================================================================
def process_game_with_stats(pbp_items, home_starters, away_starters, player_team_map, home_team_name, away_team_name):
    current_home = set(home_starters)
    current_away = set(away_starters)
    stints_log = []
    stint_start_clock = "20:00"
    
    running_home_score = 0
    running_away_score = 0
    
    stats = {
        "home": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0},
        "away": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0}
    }
    
    all_players = list(player_team_map.keys())
    
    for play in pbp_items:
        text = play.get('text', '')
        clock = play.get('clock', {}).get('displayValue', '0:00')
        text_lower = text.lower()
        period = play.get('period', {}).get('number', 1)
        
        act_team = None
        for player in all_players:
            if player in text:
                act_team = player_team_map[player]
                break
        
        if not act_team:
            if away_team_name in text_lower: act_team = "away"
            elif home_team_name in text_lower: act_team = "home"

        if act_team:
            is_shot = any(kw in text_lower for kw in ["shot", "layup", "jumper", "dunk", "three-pointer", "3-pointer", "free throw"])
            is_make = any(kw in text_lower for kw in ["makes", "made", "good"])
            is_miss = any(kw in text_lower for kw in ["misses", "missed"])
            
            if is_shot:
                if "free throw" not in text_lower:
                    stats[act_team]["fga"] += 1
                if is_make:
                    if "free throw" in text_lower:
                        stats[act_team]["pts"] += 1
                        running_home_score += (1 if act_team == "home" else 0)
                        running_away_score += (1 if act_team == "away" else 0)
                        stats[act_team]["fta"] += 1
                    elif "three-point" in text_lower or "3-point" in text_lower or "three-pointer" in text_lower:
                        stats[act_team]["pts"] += 3
                        running_home_score += (3 if act_team == "home" else 0)
                        running_away_score += (3 if act_team == "away" else 0)
                    else:
                        stats[act_team]["pts"] += 2
                        running_home_score += (2 if act_team == "home" else 0)
                        running_away_score += (2 if act_team == "away" else 0)
                elif is_miss and "free throw" in text_lower:
                    stats[act_team]["fta"] += 1
            elif any(kw in text_lower for kw in ["turnover", "bad pass", "lost ball", "traveling", "offensive foul"]):
                stats[act_team]["tov"] += 1
            elif "offensive rebound" in text_lower or "off rebound" in text_lower:
                stats[act_team]["orb"] += 1

        if "subbing in" in text_lower or "subbing out" in text_lower or "enters the game" in text_lower:
            if check_kenpom_garbage_time(clock, period, running_home_score, running_away_score):
                break
                
            home_poss = calculate_possessions(stats["home"])
            away_poss = calculate_possessions(stats["away"])
            
            stints_log.append({
                "start_clock": stint_start_clock,
                "end_clock": clock,
                "home_lineup": list(current_home),
                "away_lineup": list(current_away),
                "stats": copy.deepcopy(stats), 
                "home_possessions": home_poss,
                "away_possessions": away_poss
            })
            
            match_in = re.search(r"(.+?)\s+(?:subbing in|enters the game)", text)
            match_out = re.search(r"(.+?)\s+(?:subbing out|leaves the game)", text)
            
            if match_in and act_team:
                p_in = match_in.group(1).strip()
                if act_team == "home": current_home.add(p_in)
                else: current_away.add(p_in)
            if match_out:
                p_out = match_out.group(1).strip()
                current_home.discard(p_out)
                current_away.discard(p_out)
            
            stats = {
                "home": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0},
                "away": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0}
            }
            stint_start_clock = clock
            
    return stints_log


# =====================================================================
# 4. ISOLATED THREAD WORKER FUNCTION
# =====================================================================
def process_individual_game(game_id):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        home_l, away_l, player_map, home_team, away_team = get_game_data_and_player_map(game_id)
        if not player_map:
            return None
        
        pbp_url = f"https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/events/{game_id}/competitions/{game_id}/plays?limit=1000"
        pbp_response = requests.get(pbp_url, headers=headers, timeout=12)
        
        if pbp_response.status_code == 429:
            time.sleep(2)
            pbp_response = requests.get(pbp_url, headers=headers, timeout=12)
            
        if pbp_response.status_code != 200:
            return {"status": "error", "reason": f"HTTP {pbp_response.status_code}"}
            
        raw_plays_list = pbp_response.json().get('items', [])
        calculated_stints = process_game_with_stats(raw_plays_list, home_l, away_l, player_map, home_team, away_team)
        
        local_stats = defaultdict(lambda: {"off_pts": 0, "off_poss": 0, "def_pts": 0, "def_poss": 0})
        game_active_players = set()
        local_teams = {}
        
        for stint in calculated_stints:
            home_lineup = stint["home_lineup"]
            away_lineup = stint["away_lineup"]
            home_pts = stint["stats"]["home"]["pts"]
            away_pts = stint["stats"]["away"]["pts"]
            home_poss = stint["home_possessions"]
            away_poss = stint["away_possessions"]
            
            for player in home_lineup:
                if not player or player == "Unknown" or len(player) < 3: continue
                local_stats[player]["off_pts"] += home_pts
                local_stats[player]["off_poss"] += home_poss
                local_stats[player]["def_pts"] += away_pts
                local_stats[player]["def_poss"] += away_poss
                game_active_players.add(player)
                local_teams[player] = home_team.title()
                
            for player in away_lineup:
                if not player or player == "Unknown" or len(player) < 3: continue
                local_stats[player]["off_pts"] += away_pts
                local_stats[player]["off_poss"] += away_poss
                local_stats[player]["def_pts"] += home_pts
                local_stats[player]["def_poss"] += home_poss
                game_active_players.add(player)
                local_teams[player] = away_team.title()
                
        return {
            "status": "success",
            "game_id": game_id,
            "stats": dict(local_stats),
            "active_players": list(game_active_players),
            "teams": local_teams
        }
    except Exception as e:
        return {"status": "error", "reason": str(e)}


# =====================================================================
# 5. PARALLEL TIMELINE EXECUTION PIPELINE
# =====================================================================
def run_parallel_season_analysis(start_date_str, end_date_str, max_workers=15, output_csv="season_cumulative_player_efficiency.csv"):
    # The master dictionary remains local to the main thread
    master_player_totals = defaultdict(lambda: {"off_pts": 0, "off_poss": 0, "def_pts": 0, "def_poss": 0, "games_played": 0})
    player_team_registry = {}
    
    start_dt = datetime.strptime(start_date_str, "%Y%m%d")
    end_dt = datetime.strptime(end_date_str, "%Y%m%d")
    current_dt = start_dt
    
    print("Step 1: Harvesting all scheduled Game IDs across calendar window...")
    all_game_ids = []
    while current_dt <= end_dt:
        date_query = current_dt.strftime("%Y%m%d")
        day_ids = get_game_ids_for_date(date_query)
        all_game_ids.extend(day_ids)
        current_dt += timedelta(days=1)
        
    total_discovered_games = len(all_game_ids)
    print(f" -> Found {total_discovered_games} total STRICT D1 vs D1 games to analyze.")
    print(f"\nStep 2: Launching parallel parsing pool using {max_workers} worker threads...")
    print("-" * 80)
    
    total_processed_games = 0
    total_skipped_games = 0
    completed_futures_count = 0
    
    # Fire up workers. They only read web data and return isolated local dictionaries.
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_game = {executor.submit(process_individual_game, gid): gid for gid in all_game_ids}
        
        # Centralized Reducer Loop: Only the main execution thread iterates here
        for future in concurrent.futures.as_completed(future_to_game):
            completed_futures_count += 1
            result = future.result()
            
            if result and result.get("status") == "success":
                total_processed_games += 1
                
                # Fetch team mapping context built natively by the isolated thread
                game_teams = result["teams"] 
                
                # Apply updates linearily without thread overlap risk
                for player, stats in result["stats"].items():
                    team_name = game_teams.get(player, "Unknown Team")
                    unique_key = f"{player} ({team_name})"
                    
                    master_player_totals[unique_key]["off_pts"] += stats["off_pts"]
                    master_player_totals[unique_key]["off_poss"] += stats["off_poss"]
                    master_player_totals[unique_key]["def_pts"] += stats["def_pts"]
                    master_player_totals[unique_key]["def_poss"] += stats["def_poss"]
                    player_team_registry[unique_key] = team_name
                    
                for player in result["active_players"]:
                    team_name = game_teams.get(player, "Unknown Team")
                    unique_key = f"{player} ({team_name})"
                    master_player_totals[unique_key]["games_played"] += 1
            else:
                total_skipped_games += 1
            
            if completed_futures_count % 100 == 0 or completed_futures_count == total_discovered_games:
                pct = (completed_futures_count / total_discovered_games) * 100
                print(f"Progress: {completed_futures_count}/{total_discovered_games} processed ({pct:.1f}%) | Success: {total_processed_games} | Dropped: {total_skipped_games}")
        
    print("\nStep 3: Flattening accumulated metrics into true seasonal weighted volume value...")
    rows = []
    for unique_key, s in master_player_totals.items():
        # Keep players who played at least a tiny sliver of a possession on BOTH ends
        if s["off_poss"] <= 0 or s["def_poss"] <= 0: 
            continue
            
        display_name = unique_key.split(" (")[0]
        
        # FIX: Explicit try-except blocks completely eliminate line-level ZeroDivisionErrors
        try:
            ortg = (s["off_pts"] / s["off_poss"] * 100)
        except ZeroDivisionError:
            ortg = 0.0
            
        try:
            drtg = (s["def_pts"] / s["def_poss"] * 100)
        except ZeroDivisionError:
            drtg = 0.0
        
        net_efficiency = ortg - drtg
        
        # Calculate Net Points Produced (Volume x Efficiency)
        net_points_produced = (net_efficiency * s["off_poss"]) / 100
        
        rows.append({
            "Player": display_name,
            "Assigned_Team": player_team_registry.get(unique_key, "N/A"),
            "Games_Tracked": s["games_played"],
            "Season_Off_Possessions": round(s["off_poss"], 1),
            "Weighted_ORtg": round(ortg, 1),
            "Season_Def_Possessions": round(s["def_poss"], 1),
            "Weighted_DRtg": round(drtg, 1),
            "Net_Efficiency_Rating": round(net_efficiency, 1),
            "Net_Points_Produced": round(net_points_produced, 1)
        })
        
    df_season = pd.DataFrame(rows)
    if not df_season.empty:
        # Sort by Net Points Produced so volume operators dominate the top spots
        df_season = df_season.sort_values(by="Net_Points_Produced", ascending=False).reset_index(drop=True)
        df_season.to_csv(output_csv, index=False)
        print("="*80)
        print(f"[SUCCESS] Compilation complete using Volume-Weighted Value.")
        print(f"Total Unique Players Saved: {len(df_season)}")
        print(f"Master Sheet Location: '{output_csv}'")
        print("="*80)


# =====================================================================
# 6. EXECUTION BLOCK
# =====================================================================
if __name__ == "__main__":
    START_DATE = "20251103"
    END_DATE = "20260406"
    
    WORKER_THREADS = 15 
    
    run_parallel_season_analysis(
        START_DATE, 
        END_DATE, 
        max_workers=WORKER_THREADS, 
        output_csv="player_3.csv"
    )

Step 1: Harvesting all scheduled Game IDs across calendar window...
 -> Found 5767 total STRICT D1 vs D1 games to analyze.

Step 2: Launching parallel parsing pool using 15 worker threads...
--------------------------------------------------------------------------------
Progress: 100/5767 processed (1.7%) | Success: 100 | Dropped: 0
Progress: 200/5767 processed (3.5%) | Success: 200 | Dropped: 0
Progress: 300/5767 processed (5.2%) | Success: 300 | Dropped: 0
Progress: 400/5767 processed (6.9%) | Success: 400 | Dropped: 0
Progress: 500/5767 processed (8.7%) | Success: 500 | Dropped: 0
Progress: 600/5767 processed (10.4%) | Success: 600 | Dropped: 0
Progress: 700/5767 processed (12.1%) | Success: 700 | Dropped: 0
Progress: 800/5767 processed (13.9%) | Success: 800 | Dropped: 0
Progress: 900/5767 processed (15.6%) | Success: 900 | Dropped: 0
Progress: 1000/5767 processed (17.3%) | Success: 1000 | Dropped: 0
Progress: 1100/5767 processed (19.1%) | Success: 1100 | Dropped: 0
Progress: 1200

In [36]:
import pandas as pd
import numpy as np

df = pd.read_csv("player_3.csv")
#df.sort_values(by = "Games_Tracked", ascending = False)
df["Balanced"] = round((df["Weighted_ORtg"] * df["Season_Off_Possessions"] / 100) - (df["Weighted_DRtg"] * df["Season_Def_Possessions"] / 100),1)
df["Star"] = round((df["Weighted_ORtg"] - df["Weighted_DRtg"]) * df["Season_Off_Possessions"] / 100, 1)
df["Defense"] = round((df["Weighted_ORtg"] - df["Weighted_DRtg"]) * np.sqrt(df["Season_Def_Possessions"] * df["Season_Off_Possessions"] / 100), 1)
df["WAR"] = round(((df["Weighted_ORtg"] - 100) * df["Season_Off_Possessions"] / 100) + ((100 - df["Weighted_DRtg"]) * df["Season_Def_Possessions"] / 100), 1)
df[df["Assigned_Team"] == "Iowa State Cyclones"]
#df.sort_values(by = "Defense", ascending = False)
#df.to_csv('player_4.csv')
#df

,Player,Assigned_Team,Games_Tracked,Season_Off_Possessions,Weighted_ORtg,Season_Def_Possessions,Weighted_DRtg,Net_Efficiency_Rating,Net_Points_Produced,Balanced,Star,Defense,WAR
12,Joshua Jefferson,Iowa State Cyclones,35,1648.1,102.8,1584.5,81.5,21.4,352.2,402.9,351.0,3442.1,339.3
24,Milan Momcilovic,Iowa State Cyclones,37,1707.7,101.0,1657.0,82.6,18.4,314.1,356.1,314.2,3095.2,305.4
32,Killyan Toure,Iowa State Cyclones,37,1394.1,102.9,1370.0,81.4,21.5,299.4,319.3,299.7,2971.3,295.2
47,Blake Buchanan,Iowa State Cyclones,37,1352.0,103.1,1330.2,82.9,20.2,272.9,291.2,273.1,2708.9,269.4
60,Tamin Lipsey,Iowa State Cyclones,34,1659.9,99.6,1584.5,84.4,15.3,253.4,315.9,252.3,2465.1,240.5
66,Nate Heise,Iowa State Cyclones,37,1395.4,101.5,1361.0,83.8,17.7,247.2,275.8,247.0,2439.2,241.4
241,Jamarion Batemon,Iowa State Cyclones,37,836.3,100.6,800.5,82.1,18.5,154.7,184.1,154.7,1513.7,148.3
434,Dominykas Pleta,Iowa State Cyclones,35,610.7,101.5,580.7,83.7,17.8,108.9,133.8,108.7,1060.0,103.8
818,Dominick Nelson,Iowa State Cyclones,17,265.5,107.7,256.5,85.8,21.9,58.2,65.9,58.1,571.5,56.9
1381,Cade Kelderman,Iowa State Cyclones,2,23.3,158.6,23.9,75.2,83.5,19.5,19.0,19.4,196.8,19.6


In [19]:
import requests
import re
import copy
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import scipy.sparse as sparse

# =====================================================================
# 1. DIVISION I AUTOMATED FILTER ENGINE
# =====================================================================
def fetch_division_1_team_set():
    print("[INIT] Syncing official NCAA Division-I team directory...")
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams?limit=1000"
    headers = {"User-Agent": "Mozilla/5.0"}
    d1_teams = set()
    
    try:
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code == 200:
            sports = res.json().get("sports", [])
            if sports:
                leagues = sports[0].get("leagues", [])
                if leagues:
                    teams_pool = leagues[0].get("teams", [])
                    for t in teams_pool:
                        team_node = t.get("team", {})
                        short_name = team_node.get("shortDisplayName")
                        if short_name:
                            d1_teams.add(short_name)
    except Exception as e:
        print(f"[WARNING] Team directory sync failed: {e}.")
        
    print(f"[SUCCESS] Isolated {len(d1_teams)} official Division-I programs.")
    return d1_teams

# =====================================================================
# 2. STATISTICAL HELPERS
# =====================================================================
def calculate_possessions(s):
    return s["fga"] + (0.475 * s["fta"]) + s["tov"] - s["orb"]

def check_kenpom_garbage_time(clock_string, period, home_score, away_score):
    try:
        minutes, seconds = map(int, clock_string.split(":"))
        half_mins_remaining = minutes + (seconds / 60.0)
        mins_remaining = half_mins_remaining + 20.0 if period == 1 else half_mins_remaining
    except:
        return False
        
    margin = abs(home_score - away_score)
    if mins_remaining > 10.0 and margin >= 30: return True
    if 5.0 <= mins_remaining <= 10.0 and margin >= 25: return True
    if 2.0 <= mins_remaining < 5.0 and margin >= 20: return True
    if mins_remaining < 2.0 and margin >= 11: return True
    return False

def generate_season_dates(start_date_str, end_date_str):
    start = datetime.strptime(start_date_str, "%m/%d/%Y")
    end = datetime.strptime(end_date_str, "%m/%d/%Y")
    date_list = []
    current = start
    while current <= end:
        date_list.append(current.strftime("%Y%m%d"))
        current += timedelta(days=1)
    return date_list

def process_single_game(game_id, d1_team_set):
    headers = {"User-Agent": "Mozilla/5.0"}
    stints_collected = []
    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary?event={game_id}"
    
    try:
        res = requests.get(url, headers=headers, timeout=8)
        if res.status_code != 200: return []
        data = res.json()
    except:
        return []

    box_data = data.get('boxscore', {})
    players_list = box_data.get('players', [])
    teams_info = box_data.get('teams', [])
    
    if not players_list or len(teams_info) < 2: return []
    
    away_team_name = teams_info[0].get('team', {}).get('shortDisplayName', 'Away')
    home_team_name = teams_info[1].get('team', {}).get('shortDisplayName', 'Home')
    
    if d1_team_set and ((away_team_name not in d1_team_set) or (home_team_name not in d1_team_set)):
        return []

    home_starters, away_starters = [], []
    player_team_map = {}
    player_box_metrics = {}
    
    for index, team_data in enumerate(players_list):
        team_side = "home" if index == 1 else "away"
        team_text_name = home_team_name if index == 1 else away_team_name
        
        stats_node = team_data.get('statistics', [{}])[0]
        labels = stats_node.get('labels', [])
        athletes = stats_node.get('athletes', [])
        
        # Safely find label indexes to prevent ValueError
        def get_index(possible_names):
            for name in possible_names:
                if name in labels: return labels.index(name)
            return -1
            
        pts_idx = get_index(['PTS'])
        ast_idx = get_index(['AST'])
        reb_idx = get_index(['REB', 'TOT'])
        stl_idx = get_index(['STL'])
        blk_idx = get_index(['BLK'])
        to_idx  = get_index(['TO', 'TOV'])
        fg_idx  = get_index(['FG'])
            
        for athlete in athletes:
            name = athlete.get('athlete', {}).get('displayName', 'Unknown')
            if name == 'Unknown': continue
            
            player_team_map[name] = (team_side, team_text_name)
            if athlete.get('starter') == True:
                if index == 1: home_starters.append(name)
                else: away_starters.append(name)
                
            raw_stats = athlete.get('stats', [])
            
            # --- FIX 1: Bulletproof stat extraction ---
            def safe_stat(idx):
                if idx >= 0 and idx < len(raw_stats):
                    val = str(raw_stats[idx])
                    if '-' in val: val = val.split('-')[0] # Grab makes from "FGM-FGA"
                    if val.replace('.','',1).isdigit(): return float(val)
                return 0.0
                
            pts = safe_stat(pts_idx)
            ast = safe_stat(ast_idx)
            reb = safe_stat(reb_idx)
            stl = safe_stat(stl_idx)
            blk = safe_stat(blk_idx)
            tov = safe_stat(to_idx)
            
            # Extract Field Goals Attempted for efficiency penalty
            fga = 0.0
            if fg_idx >= 0 and fg_idx < len(raw_stats):
                fg_val = str(raw_stats[fg_idx])
                if '-' in fg_val and fg_val.split('-')[1].isdigit():
                    fga = float(fg_val.split('-')[1])
                    
            # --- FIX 2: Comprehensive Box Prior to shatter Teammate Clumping ---
            box_value = pts + (0.7 * ast) + (0.5 * reb) + (1.0 * stl) + (0.8 * blk) - (1.0 * tov) - (0.4 * fga)
            
            player_box_metrics[f"{name} || {team_text_name}"] = {
                "box_value": box_value,
                "games": 1
            }

    pbp_items = data.get('plays', [])
    if not pbp_items: return []

    current_home = set(home_starters)
    current_away = set(away_starters)
    running_home_score, running_away_score = 0, 0
    
    stats = {"home": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0}, "away": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0}}
    all_players = list(player_team_map.keys())
    
    for play in pbp_items:
        text = play.get('text', '')
        clock = play.get('clock', {}).get('displayValue', '0:00')
        text_lower = text.lower()
        period = play.get('period', {}).get('number', 1)
        
        act_team = None
        for player in all_players:
            if player in text:
                mapping = player_team_map.get(player, (None, None))
                if mapping[0]: act_team = mapping[0]
                break

        if act_team:
            is_shot = any(kw in text_lower for kw in ["shot", "layup", "jumper", "dunk", "three-pointer", "3-pointer", "free throw"])
            is_make = any(kw in text_lower for kw in ["makes", "made", "good"])
            is_miss = any(kw in text_lower for kw in ["misses", "missed"])
            
            if is_shot:
                if "free throw" not in text_lower: stats[act_team]["fga"] += 1
                if is_make:
                    if "free throw" in text_lower:
                        stats[act_team]["pts"] += 1
                        running_home_score += (1 if act_team == "home" else 0)
                        running_away_score += (1 if act_team == "away" else 0)
                        stats[act_team]["fta"] += 1
                    elif any(kw in text_lower for kw in ["three-point", "3-point", "three-pointer"]):
                        stats[act_team]["pts"] += 3
                        running_home_score += (3 if act_team == "home" else 0)
                        running_away_score += (3 if act_team == "away" else 0)
                    else:
                        stats[act_team]["pts"] += 2
                        running_home_score += (2 if act_team == "home" else 0)
                        running_away_score += (2 if act_team == "away" else 0)
                elif is_miss and "free throw" in text_lower:
                    stats[act_team]["fta"] += 1
            elif any(kw in text_lower for kw in ["turnover", "bad pass", "lost ball", "traveling", "offensive foul"]):
                stats[act_team]["tov"] += 1
            elif "offensive rebound" in text_lower or "off rebound" in text_lower:
                stats[act_team]["orb"] += 1

        if "subbing in" in text_lower or "subbing out" in text_lower or "enters the game" in text_lower:
            if check_kenpom_garbage_time(clock, period, running_home_score, running_away_score):
                break
                
            home_poss = calculate_possessions(stats["home"])
            away_poss = calculate_possessions(stats["away"])
            
            home_lineup_clean = [f"{p} || {player_team_map.get(p, (None, 'Unknown Team'))[1]}" for p in current_home]
            away_lineup_clean = [f"{p} || {player_team_map.get(p, (None, 'Unknown Team'))[1]}" for p in current_away]
            
            stints_collected.append({
                "game_id": game_id,
                "home_team": home_team_name,
                "away_team": away_team_name,
                "home_lineup": home_lineup_clean,
                "away_lineup": away_lineup_clean,
                "stats": copy.deepcopy(stats), 
                "home_possessions": home_poss,
                "away_possessions": away_poss
            })
            
            match_in = re.search(r"(.+?)\s+(?:subbing in|enters the game)", text)
            match_out = re.search(r"(.+?)\s+(?:subbing out|leaves the game)", text)
            
            if match_in and act_team:
                p_in = match_in.group(1).strip()
                if act_team == "home": current_home.add(p_in)
                else: current_away.add(p_in)
            if match_out:
                p_out = match_out.group(1).strip()
                current_home.discard(p_out)
                current_away.discard(p_out)
            
            stats = {"home": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0}, "away": {"pts": 0, "fga": 0, "fta": 0, "tov": 0, "orb": 0}}
            
    return stints_collected, player_box_metrics

# =====================================================================
# 4. FIXED TEAM EFFECTS & DESIGN MATRIX ADVANCEMENT
# =====================================================================
def build_global_rapm_matrices(all_game_stints):
    unique_players = set()
    unique_teams = set()
    player_poss_dict = {}
    
    for stint in all_game_stints:
        unique_teams.add(stint["home_team"])
        unique_teams.add(stint["away_team"])
        home_poss = stint["home_possessions"]
        away_poss = stint["away_possessions"]
        
        for p in stint["home_lineup"]:
            unique_players.add(p)
            player_poss_dict[p] = player_poss_dict.get(p, 0) + home_poss
        for p in stint["away_lineup"]:
            unique_players.add(p)
            player_poss_dict[p] = player_poss_dict.get(p, 0) + away_poss
            
    all_players = sorted(list(unique_players))
    all_teams = sorted(list(unique_teams))
    
    player_to_idx = {player: idx for idx, player in enumerate(all_players)}
    team_to_idx = {team: idx for idx, team in enumerate(all_teams)}
    
    num_stints = len(all_game_stints)
    num_players = len(all_players)
    num_teams = len(all_teams)
    
    total_cols = (num_players * 2) + (num_teams * 2)
    X_sparse = sparse.lil_matrix((num_stints * 2, total_cols), dtype=np.float32)
    targets = np.zeros(num_stints * 2, dtype=np.float32)
    weights = np.zeros(num_stints * 2, dtype=np.float32)
    
    valid_row_idx = 0
    for stint in all_game_stints:
        home_poss = stint["home_possessions"]
        away_poss = stint["away_possessions"]
        home_pts = stint["stats"]["home"]["pts"]
        away_pts = stint["stats"]["away"]["pts"]
        
        h_team_idx = team_to_idx[stint["home_team"]]
        a_team_idx = team_to_idx[stint["away_team"]]
        
        if home_poss > 0:
            home_ortg = (home_pts / home_poss) * 100
            for p in stint["home_lineup"]: X_sparse[valid_row_idx, player_to_idx[p]] = 1.0
            for p in stint["away_lineup"]: X_sparse[valid_row_idx, num_players + player_to_idx[p]] = 1.0
            X_sparse[valid_row_idx, (num_players * 2) + h_team_idx] = 1.0  
            X_sparse[valid_row_idx, (num_players * 2) + num_teams + a_team_idx] = 1.0  
            targets[valid_row_idx] = home_ortg
            weights[valid_row_idx] = home_poss
            valid_row_idx += 1
            
        if away_poss > 0:
            away_ortg = (away_pts / away_poss) * 100
            for p in stint["away_lineup"]: X_sparse[valid_row_idx, player_to_idx[p]] = 1.0
            for p in stint["home_lineup"]: X_sparse[valid_row_idx, num_players + player_to_idx[p]] = 1.0
            X_sparse[valid_row_idx, (num_players * 2) + a_team_idx] = 1.0  
            X_sparse[valid_row_idx, (num_players * 2) + num_teams + h_team_idx] = 1.0  
            targets[valid_row_idx] = away_ortg
            weights[valid_row_idx] = away_poss
            valid_row_idx += 1
            
    X_sparse = X_sparse[:valid_row_idx, :].tocsr()
    targets = targets[:valid_row_idx]
    weights = weights[:valid_row_idx]
        
    return X_sparse, targets, weights, all_players, all_teams, player_poss_dict

# =====================================================================
# 5. PRIOR-INFORMED RE-CENTERED MATRIX SOLVER
# =====================================================================
def train_global_rapm_model(X, Y, W, player_tokens, team_tokens, player_poss_dict, player_box_totals, alpha=2500):
    num_players = len(player_tokens)
    num_teams = len(team_tokens)
    num_rows = X.shape[0]
    
    player_prior_O = np.zeros(num_players, dtype=np.float32)
    player_prior_D = np.zeros(num_players, dtype=np.float32)
    
    for idx, token in enumerate(player_tokens):
        # We are pulling the newly calculated comprehensive box value
        box_data = player_box_totals.get(token, {"box_value": 0, "games": 1})
        avg_box = box_data["box_value"] / max(box_data["games"], 1)
        
        # Scale individual box score baseline prior linearly
        # Elite players (avg_box > 15) get massive priors, role players drop to 0 or negatives
        player_prior_O[idx] = (avg_box * 0.35) - 2.0
        player_prior_D[idx] = (avg_box * 0.15) - 0.5
        
    Y_residual = np.copy(Y)
    print(" -> Formulating mathematical residual matrices...")
    
    
    # Pre-calculate matrix shift offsets to remove prior bias cleanly from linear target vector
    for row_idx in range(num_rows):
        # Scan non-zero coordinates inside player rows to compute shift values
        row_start = X.indptr[row_idx]
        row_end = X.indptr[row_idx + 1]
        
        for i in range(row_start, row_end):
            col_idx = X.indices[i]
            if col_idx < num_players: # Player Offense column
                Y_residual[row_idx] -= player_prior_O[col_idx]
            elif num_players <= col_idx < num_players * 2: # Player Defense column
                Y_residual[row_idx] -= player_prior_D[col_idx - num_players]

    intercept_col = sparse.csc_matrix(np.ones((num_rows, 1), dtype=np.float32))
    X_with_intercept = sparse.hstack([X, intercept_col], format="csr")
    
    total_cols = X_with_intercept.shape[1]
    diag_penalties = np.zeros(total_cols, dtype=np.float32)
    
    # Apply dynamic volume penalties to avoid bench variance loops
    for idx, token in enumerate(player_tokens):
        poss = max(player_poss_dict[token], 1.0)
        dynamic_alpha = alpha * (1500.0 / poss)
        dynamic_alpha = min(dynamic_alpha, alpha * 12.0)
        
        diag_penalties[idx] = dynamic_alpha                     
        diag_penalties[num_players + idx] = dynamic_alpha       
        
    diag_penalties[num_players * 2 : num_players * 2 + num_teams * 2] = 10.0   
    diag_penalties[-1] = 0.0 # Free global intercept constant
    
    print(" -> Executing Normal Equations matrix maps...")
    X_weighted = X_with_intercept.multiply(W[:, None])
    A = X_weighted.T.dot(X_with_intercept)
    A.setdiag(A.diagonal() + diag_penalties)
    B = X_weighted.T.dot(Y_residual)
    coefficients = sparse.linalg.spsolve(A, B)
    
    # 3. Add individual player priors back to finished residuals
    offensive_coefs = coefficients[:num_players] + player_prior_O
    defensive_coefs = coefficients[num_players:num_players * 2] + player_prior_D
    
    team_off_baselines = coefficients[num_players * 2 : num_players * 2 + num_teams]
    team_def_baselines = coefficients[num_players * 2 + num_teams : num_players * 2 + num_teams * 2]
    
    global_intercept = coefficients[-1]
    print(f" -> Global standard intercept baseline: {global_intercept:.2f}")
    
    team_env_df = pd.DataFrame({
        "Team": team_tokens,
        "Team_Off_Env": team_off_baselines,
        "Team_Def_Env": team_def_baselines * -1
    })
    
    names, teams, possessions = [], [], []
    for token in player_tokens:
        parts = token.rsplit(" || ", 1)
        names.append(parts[0] if len(parts) == 2 else token)
        teams.append(parts[1] if len(parts) == 2 else "Unknown Team")
        possessions.append(np.round(player_poss_dict[token], 1))
        
    df = pd.DataFrame({
        "Player": names,
        "Team": teams,
        "Est_Possessions": possessions,
        "O_BAPM": offensive_coefs,
        "D_BAPM": defensive_coefs * -1
    })
    
    df = df.merge(team_env_df, on="Team", how="left")
    
    df["O_BAPM"] = np.round(df["O_BAPM"] + df["Team_Off_Env"], 2)
    df["D_BAPM"] = np.round(df["D_BAPM"] + df["Team_Def_Env"], 2)
    df["Total_BAPM"] = np.round(df["O_BAPM"] + df["D_BAPM"], 2)
    
    return df[["Player", "Team", "Est_Possessions", "O_BAPM", "D_BAPM", "Total_BAPM"]]

# =====================================================================
# 6. SCHEDULER CONTROLLER
# =====================================================================
def run_high_speed_season_pipeline(start_date, end_date, max_workers=20, ridge_alpha=1200):
    d1_team_set = fetch_division_1_team_set()
    date_list = generate_season_dates(start_date, end_date)
    headers = {"User-Agent": "Mozilla/5.0"}
    
    print(f"\nGathering schedules across {len(date_list)} days...")
    all_completed_game_ids = []
    
    for date_str in date_list:
        scoreboard_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard?dates={date_str}&groups=50&limit=500"
        try:
            res = requests.get(scoreboard_url, headers=headers, timeout=5)
            if res.status_code == 200:
                events = res.json().get("events", [])
                for e in events:
                    if e.get("status", {}).get("type", {}).get("name") == "STATUS_FINAL":
                        all_completed_game_ids.append(e.get("id"))
        except:
            continue
            
    total_games = len(all_completed_game_ids)
    print(f"[SCHEDULE COMPLETE] Found {total_games} total matches.")
    if total_games == 0: return

    master_stints_pool = []
    master_box_totals = {}
    start_time = time.time()
    
    print(f"\nSpinning up high-efficiency matrix scheduler ({max_workers} threads)...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_game = {executor.submit(process_single_game, g_id, d1_team_set): g_id for g_id in all_completed_game_ids}
        completed_count = 0
        for future in as_completed(future_to_game):
            completed_count += 1
            
            try:
                worker_data = future.result()
                if worker_data: 
                    result, game_box = worker_data
                    
                    if result:
                        master_stints_pool.extend(result)
                        for p_token, stats in game_box.items():
                            if p_token not in master_box_totals:
                                master_box_totals[p_token] = {"box_value": 0.0, "games": 0} # Updated here
                            master_box_totals[p_token]["box_value"] += stats["box_value"]   # Updated here
                            master_box_totals[p_token]["games"] += stats["games"]
            except Exception as e:
                pass
                
            if completed_count % 50 == 0 or completed_count == total_games:
                elapsed = time.time() - start_time
                print(f" -> Compiled [{completed_count}/{total_games}] games... Total Stints: {len(master_stints_pool)} (Elapsed: {elapsed:.1f}s)", end="\r")

    print(f"\n\n[SCRAPE DONE] Harvest completed in {time.time() - start_time:.1f} seconds.")
    if len(master_stints_pool) == 0: 
        print("[ABORT] No stints were retained. Check parser logic.")
        return

    print("\n[SPARSE INIT] Constructing Multi-Layered SOS Sparse Matrices...")
    X, Y, W, player_tokens, team_tokens, player_poss = build_global_rapm_matrices(master_stints_pool)
    
    print(f"[TRAINING] Solving Hybrid Box-Score Prior Informed Model...")
    season_leaderboard = train_global_rapm_model(X, Y, W, player_tokens, team_tokens, player_poss, master_box_totals, alpha=ridge_alpha)
    
    season_leaderboard = season_leaderboard.sort_values(by="Total_BAPM", ascending=False).reset_index(drop=True)
    
    output_file = "final_season_bapm_ratings.csv"
    season_leaderboard.to_csv(output_file, index=False)
    print(f"\n[SUCCESS] Box Score Prior-Informed BAPM File written to '{output_file}'")
    
    print("\nTop 15 Individual Value Leaders:")
    print(season_leaderboard.head(15).to_string(index=False))


if __name__ == "__main__":
    run_high_speed_season_pipeline(start_date="11/4/2024", end_date="4/10/2025", max_workers=20, ridge_alpha=2500)

[INIT] Syncing official NCAA Division-I team directory...
[SUCCESS] Isolated 362 official Division-I programs.

Gathering schedules across 158 days...
[SCHEDULE COMPLETE] Found 6292 total matches.

Spinning up high-efficiency matrix scheduler (20 threads)...
 -> Compiled [6292/6292] games... Total Stints: 182695 (Elapsed: 81.1s)

[SCRAPE DONE] Harvest completed in 81.1 seconds.

[SPARSE INIT] Constructing Multi-Layered SOS Sparse Matrices...
[TRAINING] Solving Hybrid Box-Score Prior Informed Model...
 -> Formulating mathematical residual matrices...
 -> Executing Normal Equations matrix maps...
 -> Global standard intercept baseline: 83.21

[SUCCESS] Box Score Prior-Informed BAPM File written to 'final_season_bapm_ratings.csv'

Top 15 Individual Value Leaders:
            Player    Team  Est_Possessions    O_BAPM    D_BAPM  Total_BAPM
      Cooper Flagg    Duke            633.6 25.410000 16.510000   41.919998
      Kon Knueppel    Duke            815.4 24.440001 16.959999   41.400002
 

In [12]:
import pandas as pd
df = pd.read_csv('final_season_bapm_ratings.csv')
df.head()

,Player,Team,O_BAPM,D_BAPM,Total_BAPM
0,Sion James,Duke,3.44,3.77,7.21
1,Kon Knueppel,Duke,3.49,2.97,6.46
2,Joseph Tugler,Houston,1.15,5.08,6.23
3,Cooper Flagg,Duke,3.50,1.96,5.46
4,Khalif Battle,Gonzaga,3.24,1.70,4.94


In [13]:
total = df.groupby("Team")["Total_BAPM"].sum().reset_index()
offense = df.groupby("Team")["O_BAPM"].sum().reset_index()
defense = df.groupby("Team")["D_BAPM"].sum().reset_index()
# Chain the merges together
df_merged = pd.merge(pd.merge(total, offense, on='Team', how='inner'), defense, on='Team', how='inner')
df_merged = round(df_merged, 2)
#df_merged.to_csv("team_bapm_25")
df_merged.sort_values('Total_BAPM', ascending = False).head(20)

,Team,Total_BAPM,O_BAPM,D_BAPM
70,Duke,44.21,28.21,16.00
112,Houston,27.34,8.42,18.92
101,Gonzaga,24.10,15.17,8.93
309,UC San Diego,22.32,4.42,17.90
161,McNeese,20.89,7.17,13.72
55,Colorado St,20.53,12.34,8.19
17,BYU,20.21,17.08,3.13
160,Maryland,19.12,6.60,12.52
86,Florida,18.88,16.80,2.08
279,St John's,18.24,7.66,10.58
